# Mortgage Rate of Mortgage

## Problem Statement

You are working for a mortgage company that manages various mortgage types selected by multiple users.

The company stores data in two datasets:

- MortgageDetails: Contains information about each mortgage.
- UserMortgages: Contains the mapping between users and mortgages.

Each mortgage is uniquely identified by `MortgageID`.

Each user is uniquely identified by `UserID`.

Create a function:

```python
etl(MortgageDetails, UserMortgages)
```

that returns a DataFrame containing the mortgage rate information for each mortgage type.

---

## Input Datasets

### MortgageDetails

| Column Name | Data Type |
|------------|-----------|
| MortgageID | VARCHAR |
| MortgageType | VARCHAR |
| InterestRate | DECIMAL |

### UserMortgages

| Column Name | Data Type |
|------------|-----------|
| UserID | VARCHAR |
| MortgageID | VARCHAR |

---

## Output Schema

| Column Name | Data Type |
|------------|-----------|
| MortgageType | VARCHAR |
| RateOfMortgage | DECIMAL |

---

## Requirements

- Use data from both input datasets.
- Mortgage records may be associated with multiple users.
- Mortgage records without matching details should be handled according to the problem requirements.
- Handle NULL values appropriately.
- Return the result in the expected output format.
- Output columns must be exactly:

  - MortgageType
  - RateOfMortgage

---

## Sample Input

### MortgageDetails

| MortgageID | MortgageType | InterestRate |
|------------|--------------|--------------|
| M1 | Fixed | 4.5 |
| M2 | Variable | 3.2 |
| M3 | Adjustable | 2.8 |

### UserMortgages

| UserID | MortgageID |
|---------|------------|
| U1 | M1 |
| U2 | M1 |
| U3 | M2 |
| U4 | M3 |

---

## Sample Output

| MortgageType | RateOfMortgage |
|--------------|----------------|
| Adjustable | 2.8 |
| Fixed | 4.5 |
| Variable | 3.2 |

---



In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType
from pyspark.sql.functions import *
from decimal import Decimal

# Mortgage Details
mirmortgagedetails_schema = StructType([
    StructField("MortgageID", StringType(), True),
    StructField("MortgageType", StringType(), True),
    StructField("InterestRate", DecimalType(10, 2), True)
])

mirmortgagedetails_data = [
    ("M1", "Fixed", Decimal("4.50")),
    ("M2", "Variable", Decimal("3.20")),
    ("M3", "Adjustable", Decimal("2.80"))
]

MortgageDetails = spark.createDataFrame(
    mirmortgagedetails_data,
    schema=mirmortgagedetails_schema
)

# User Mortgages
mirusermortgages_schema = StructType([
    StructField("UserID", StringType(), True),
    StructField("MortgageID", StringType(), True)
])

mirusermortgages_data = [
    ("U1", "M1"),
    ("U2", "M1"),
    ("U3", "M2"),
    ("U4", "M3")
]

UserMortgages = spark.createDataFrame(
    mirusermortgages_data,
    schema=mirusermortgages_schema
)

In [0]:
result_df = (
    MortgageDetails.join(UserMortgages, on="MortgageID")
    .groupBy("MortgageType")
    .agg(
        sum("InterestRate").alias("TotalInterestRate"),
        countDistinct("UserID").alias("TotalUsers"),
    )
    .select(
        col("MortgageType"),
        round((col("TotalInterestRate") / col("TotalUsers")), 2).alias(
            "AvgInterestRatePerUser"
        ),
    )
)
display(result_df)